# LeetCode 284 — Peeking Iterator bằng Python

Link đề: https://leetcode.com/problems/peeking-iterator/ · Độ khó: **Medium** · Chủ đề: Design, Array, Iterator

Skeleton LeetCode:

```python
# Below is the interface for Iterator, which is already defined for you.
#
# class Iterator:
#     def __init__(self, nums):
#         """
#         Initializes an iterator object to the beginning of a list.
#         :type nums: List[int]
#         """
#
#     def hasNext(self):
#         """
#         Returns true if the iteration has more elements.
#         :rtype: bool
#         """
#
#     def next(self):
#         """
#         Returns the next element in the iteration.
#         :rtype: int
#         """

class PeekingIterator:
    def __init__(self, iterator):
        """
        Initialize your data structure here.
        :type iterator: Iterator
        """

    def peek(self):
        """
        Returns the next element in the iteration without advancing the iterator.
        :rtype: int
        """

    def next(self):
        """
        :rtype: int
        """

    def hasNext(self):
        """
        :rtype: bool
        """
```

Ghi chú: đề, ví dụ và constraints lấy từ đề gốc; lớp `Iterator` do hệ thống chấm cung cấp nên ta **tự dựng** nó trong notebook (kèm bộ đếm số lần gọi). Mọi đáp án được kiểm chứng bằng code và đối chiếu với một mô hình độc lập (list + con trỏ) trên chuỗi thao tác ngẫu nhiên có seed.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề **thiết kế**: có sẵn một `Iterator` chỉ hỗ trợ `next()` (lấy phần tử kế **và tiến lên**) và `hasNext()`; hãy xây `PeekingIterator` bọc nó, bổ sung `peek()` (**xem** phần tử kế **mà không tiến lên**), vẫn giữ `next()` và `hasNext()` đúng.
2. Nhận ra **vấn đề cốt lõi**: iterator gốc không cho "xem mà không lấy"; mọi cách xem trước đều phải **lấy phần tử ra khỏi iterator gốc rồi cất lại** ở nơi nào đó (bộ đệm).
3. Đi từ **trải phẳng toàn bộ iterator vào list (tốn bộ nhớ, không dùng được cho dãy vô hạn)** → **một bộ đệm `peeked` kèm cờ (cách tham khảo)** → **bộ đệm dùng đối tượng "rỗng" riêng (đúng với mọi kiểu dữ liệu, trả lời follow up)** → **luôn lấy trước một phần tử (prefetch)** → **bộ đệm `deque` hỗ trợ xem trước `k` phần tử**.
4. Hiểu **bất biến** của lớp bọc: nếu `has_peeked` thì `peeked_element` là phần tử kế tiếp (đã bị lấy khỏi iterator gốc); `next()`, `hasNext()` phải **ưu tiên bộ đệm**.
5. Tránh các lỗi kinh điển: `peek()` làm tiến lên (hoặc gọi `next` của iterator gốc mỗi lần), `next()` bỏ qua bộ đệm (mất phần tử đã peek), `hasNext()` bỏ qua bộ đệm, và dùng `None` làm dấu "chưa peek" (sai khi dữ liệu có `None`).

## 2. Đề bài & ví dụ

Thiết kế một iterator hỗ trợ thêm thao tác `peek` trên một iterator có sẵn, ngoài `hasNext` và `next`.

Cài đặt lớp `PeekingIterator`:

- `PeekingIterator(Iterator<int> nums)`: khởi tạo đối tượng với iterator `nums` cho trước;
- `int next()`: trả về phần tử kế tiếp trong mảng và chuyển con trỏ tới phần tử kế;
- `boolean hasNext()`: trả `true` nếu còn phần tử;
- `int peek()`: trả về phần tử kế tiếp **mà không dịch con trỏ**.

Lưu ý: mỗi ngôn ngữ có cách cài đặt Iterator khác nhau, nhưng đều hỗ trợ `int next()` và `boolean hasNext()`.

**Ví dụ 1**

```
Input
["PeekingIterator", "next", "peek", "next", "next", "hasNext"]
[[[1, 2, 3]], [], [], [], [], []]
Output
[null, 1, 2, 2, 3, false]
```

Giải thích: `PeekingIterator([1,2,3])`; `next()` trả `1` (con trỏ sang `2`); `peek()` trả `2` (con trỏ **không** đổi); `next()` trả `2`; `next()` trả `3`; `hasNext()` trả `false`.

**Ràng buộc:** `1 <= nums.length <= 1000`; `1 <= nums[i] <= 1000`; mọi lời gọi `next` và `peek` đều hợp lệ (còn phần tử); tối đa `1000` lời gọi `next`, `hasNext`, `peek`.

**Follow up:** làm thế nào để mở rộng thiết kế cho **mọi kiểu dữ liệu**, không chỉ số nguyên?

In [1]:
from typing import List
import random
from collections import deque
from itertools import count


class Iterator:
    """Cài đặt giả lập của giao diện Iterator do hệ thống cung cấp (kèm bộ đếm số lần gọi next)."""

    def __init__(self, nums):
        self._nums = list(nums)
        self._pos = 0
        self.next_calls = 0                  # đếm số lần iterator gốc bị lấy phần tử

    def hasNext(self):
        return self._pos < len(self._nums)

    def next(self):
        self.next_calls += 1
        value = self._nums[self._pos]
        self._pos += 1
        return value


def run_design(cls, ops, args, make_iterator=Iterator):
    """Mô phỏng định dạng test: ops[0] là tên lớp; args[0] = [nums]."""
    obj = None
    out = []
    for position, (op, arg) in enumerate(zip(ops, args)):
        if position == 0:                       # phần tử đầu là hàm tạo (tên lớp trong ops)
            obj = cls(make_iterator(*arg))
            out.append(None)
        else:
            out.append(getattr(obj, op)(*arg))
    return out


ex_ops = ["PeekingIterator", "next", "peek", "next", "next", "hasNext"]
ex_args = [[[1, 2, 3]], [], [], [], [], []]
ex_expected = [None, 1, 2, 2, 3, False]
print("ops     =", ex_ops)
print("kỳ vọng =", ex_expected)

ops     = ['PeekingIterator', 'next', 'peek', 'next', 'next', 'hasNext']
kỳ vọng = [None, 1, 2, 2, 3, False]


## 3. Kiến thức nền

### 3.1. Iterator: lấy một phần tử là "tiêu hao" nó

`Iterator.next()` trả phần tử kế **và tiến lên**: phần tử đó không thể lấy lại từ iterator. Không có thao tác "xem rồi để nguyên". Do đó để `peek()`, ta phải **lấy** phần tử ra khỏi iterator gốc và **giữ nó lại** trong bộ nhớ của lớp bọc.

In [2]:
it = Iterator([1, 2, 3])
print(it.next(), it.next())          # 1 2
print(it.hasNext(), it.next())        # True 3
print(it.hasNext())                   # False  (không còn)

1 2
True 3
False


### 3.2. Mẫu "bộ bọc (wrapper)" và thuộc tính lưu trạng thái

`PeekingIterator` **bọc** (giữ tham chiếu) một iterator gốc `self.iterator` và thêm trạng thái của riêng mình: ở đây là **bộ đệm** chứa tối đa một phần tử đã lấy trước. Mọi thao tác công khai (`next`, `hasNext`, `peek`) đi qua bộ đệm trước khi hỏi iterator gốc.

In [3]:
class Wrapper:
    def __init__(self, inner):
        self.inner = inner               # giữ iterator gốc
        self.buffer = []                 # trạng thái riêng của lớp bọc: bộ đệm

    def peek(self):
        if not self.buffer:
            self.buffer.append(self.inner.next())      # lấy ra rồi cất vào bộ đệm
        return self.buffer[0]


w = Wrapper(Iterator([10, 20]))
print(w.peek(), w.peek(), w.inner.next_calls)             # 10 10 1: peek hai lần nhưng chỉ lấy từ iterator gốc một lần

10 10 1


### 3.3. Cờ trạng thái (`has_peeked`) hay đối tượng "rỗng" (sentinel)?

Để biết bộ đệm đang có phần tử hay không ta có hai cách: một **cờ boolean** `has_peeked` + biến `peeked_element`, hoặc một biến duy nhất `peeked` chứa một **đối tượng đặc biệt** (`_EMPTY = object()`) khi chưa có gì. Cách dùng `None` làm dấu "rỗng" **sai** khi iterator có thể trả chính `None` làm dữ liệu.

In [4]:
_EMPTY = object()                          # đối tượng duy nhất, không thể trùng với dữ liệu thật
buffer = _EMPTY
print(buffer is _EMPTY)                    # True: chưa có gì
buffer = None                              # dữ liệu hợp lệ có thể là None
print(buffer is _EMPTY, buffer is None)    # False True: phân biệt được "rỗng" với "dữ liệu None"

True
False True


### 3.4. `collections.deque` và xem trước nhiều phần tử

Nếu muốn xem trước **nhiều hơn một** phần tử (`peek(k)`), bộ đệm phải chứa nhiều phần tử: dùng `deque` (`append` cuối, `popleft` đầu, truy cập `d[i]`). Khi cần xem tới phần tử thứ `k`, lấy thêm từ iterator gốc cho đến khi bộ đệm đủ `k + 1` phần tử.

In [5]:
buf = deque()
inner = Iterator([5, 6, 7, 8])
k = 2
while len(buf) < k + 1 and inner.hasNext():
    buf.append(inner.next())
print(list(buf), "-> phần tử thứ", k, "tính từ đầu là", buf[k])

[5, 6, 7] -> phần tử thứ 2 tính từ đầu là 7


### 3.5. Dãy vô hạn và tính lười

Một iterator có thể sinh dãy **vô hạn** (ví dụ các số tự nhiên). Một lớp bọc **không được** đọc hết iterator gốc ngay khi khởi tạo (sẽ treo mãi). Chỉ lấy phần tử **khi cần** (lười) và tối đa lượng cần cho bộ đệm.

In [6]:
class Naturals:
    """Iterator vô hạn 0, 1, 2, ... (hasNext luôn True)."""

    def __init__(self):
        self.n = 0
        self.next_calls = 0

    def hasNext(self):
        return True

    def next(self):
        self.next_calls += 1
        v = self.n
        self.n += 1
        return v


inf = Naturals()
w = Wrapper(inf)
print([w.peek() for _ in range(3)], "| số lần iterator vô hạn bị lấy:", inf.next_calls)    # chỉ 1 lần

[0, 0, 0] | số lần iterator vô hạn bị lấy: 1


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Iterator gốc trên `[1, 2, 3]`. Ta muốn thực hiện: `next, peek, next, next, hasNext`.

| thao tác | iterator gốc còn | bộ đệm | giá trị trả về |
|---|---|---|---|
| (khởi tạo) | `1,2,3` | rỗng | |
| `next()` | `2,3` (đã lấy 1) | rỗng | `1` (lấy thẳng từ iterator gốc) |
| `peek()` | `3` (đã lấy 2) | `2` | `2` (lấy từ gốc, **cất vào bộ đệm**) |
| `next()` | `3` | rỗng | `2` (**lấy từ bộ đệm**, không hỏi iterator gốc) |
| `next()` | (hết) | rỗng | `3` (lấy từ iterator gốc) |
| `hasNext()` | (hết) | rỗng | `False` |

Hai điểm mấu chốt: (1) `peek()` lấy một phần tử khỏi iterator gốc và **cất vào bộ đệm**, lần `peek()` sau nếu bộ đệm còn thì **không lấy thêm**; (2) `next()` và `hasNext()` phải **nhìn bộ đệm trước**: nếu bộ đệm có phần tử thì trả bộ đệm (và xóa nó khi `next`), nếu không mới hỏi iterator gốc.

In [7]:
inner = Iterator([1, 2, 3])
buffer = []                                    # tối đa một phần tử
log = []
log.append(("next", inner.next()))             # bộ đệm rỗng: hỏi iterator gốc
if not buffer:
    buffer.append(inner.next())                # peek: lấy rồi cất
log.append(("peek", buffer[0]))
log.append(("next", buffer.pop()))             # next: lấy từ bộ đệm
log.append(("next", inner.next()))             # bộ đệm rỗng: hỏi iterator gốc
log.append(("hasNext", bool(buffer) or inner.hasNext()))
print(log)                                     # [('next', 1), ('peek', 2), ('next', 2), ('next', 3), ('hasNext', False)]

[('next', 1), ('peek', 2), ('next', 2), ('next', 3), ('hasNext', False)]


## 5. Cách 1 — Brute Force / baseline: lấy hết iterator vào một list ngay khi khởi tạo

**Ý tưởng.** Trong `__init__`, duyệt hết iterator gốc (`while hasNext(): append(next())`) vào list `items`; giữ con trỏ `pos`. `peek` là `items[pos]`, `next` là `items[pos]` rồi `pos += 1`, `hasNext` là `pos < len(items)`. Đơn giản, nhưng **tiêu thụ cả iterator ngay lập tức** và tốn `O(n)` bộ nhớ.

In [8]:
class PeekingEager:
    def __init__(self, iterator):
        self.items = []
        while iterator.hasNext():                 # lấy hết iterator gốc vào list
            self.items.append(iterator.next())
        self.pos = 0

    def peek(self):
        return self.items[self.pos]

    def next(self):
        value = self.items[self.pos]
        self.pos += 1
        return value

    def hasNext(self):
        return self.pos < len(self.items)


print(run_design(PeekingEager, ex_ops, ex_args), "| kỳ vọng", ex_expected)

[None, 1, 2, 2, 3, False] | kỳ vọng [None, 1, 2, 2, 3, False]


**Độ phức tạp (Cách 1).** Gọi `n` là số phần tử. Khởi tạo `O(n)`; `peek`, `next`, `hasNext` đều `O(1)`; **Space `O(n)`**.

**Nút thắt.** (a) Tốn `O(n)` bộ nhớ chỉ để hỗ trợ `peek`. (b) **Không dùng được với iterator vô hạn** (khởi tạo không bao giờ kết thúc). (c) Làm việc **sớm** (nếu người dùng chỉ đọc vài phần tử thì thừa). Chỉ cần **một phần tử lấy trước**.

## 6. Ý tưởng thuật toán / bước đột phá

**Một bộ đệm một phần tử là đủ.** `peek()` chỉ cần xem phần tử **kế tiếp**; ta lấy nó từ iterator gốc **đúng một lần** và cất vào bộ đệm `peeked`. Trạng thái lớp bọc gồm: `has_peeked` (bộ đệm có phần tử không) và `peeked_element`.

- `peek()`: nếu `not has_peeked`: `peeked_element = iterator.next()`, `has_peeked = True`; trả `peeked_element`.
- `next()`: nếu `has_peeked`: trả `peeked_element` và đặt `has_peeked = False`; ngược lại `iterator.next()`.
- `hasNext()`: `has_peeked or iterator.hasNext()`.

**Bất biến:** nếu `has_peeked` thì `peeked_element` là phần tử kế tiếp mà người dùng sẽ thấy, và phần tử đó **đã bị lấy khỏi iterator gốc** (không bị lấy hai lần). Mọi thao tác phải tôn trọng bộ đệm trước. Mỗi thao tác `O(1)` và bộ nhớ phụ `O(1)`.

Đối với **mọi kiểu dữ liệu**: dùng một đối tượng rỗng `_EMPTY` (hoặc cờ boolean) thay vì dựa vào `None`.

In [9]:
class Trace:
    """Bọc để in ra bộ đệm sau mỗi thao tác (minh họa)."""

    def __init__(self, iterator):
        self.iterator = iterator
        self.has_peeked = False
        self.peeked_element = None

    def peek(self):
        if not self.has_peeked:
            self.peeked_element = self.iterator.next()
            self.has_peeked = True
        return self.peeked_element

    def next(self):
        if self.has_peeked:
            self.has_peeked = False
            return self.peeked_element
        return self.iterator.next()

    def hasNext(self):
        return self.has_peeked or self.iterator.hasNext()


t = Trace(Iterator([1, 2, 3]))
for op in ("next", "peek", "peek", "next", "hasNext", "next", "hasNext"):
    print(f"{op:8} -> {getattr(t, op)()!s:6} | has_peeked={t.has_peeked}, peeked={t.peeked_element}")

next     -> 1      | has_peeked=False, peeked=None
peek     -> 2      | has_peeked=True, peeked=2
peek     -> 2      | has_peeked=True, peeked=2
next     -> 2      | has_peeked=False, peeked=2
hasNext  -> True   | has_peeked=False, peeked=2
next     -> 3      | has_peeked=False, peeked=2
hasNext  -> False  | has_peeked=False, peeked=2


## 7. Các cách tiếp theo

### Cách 2 — Bộ đệm một phần tử kèm cờ `has_peeked` (cách của lời giải tham khảo)

**Giải thích.** Đúng như mục 6, lưu `has_peeked` và `peeked_element`. `peek()` chỉ lấy từ iterator gốc nếu chưa có bộ đệm; `next()` ưu tiên bộ đệm; `hasNext()` kiểm tra bộ đệm hoặc iterator gốc.

In [10]:
class PeekingFlag:
    def __init__(self, iterator):
        self.iterator = iterator
        self.has_peeked = False
        self.peeked_element = None

    def peek(self):
        if not self.has_peeked:                       # chưa peek: lấy từ iterator gốc rồi cất
            self.peeked_element = self.iterator.next()
            self.has_peeked = True
        return self.peeked_element                     # đã peek: trả lại, KHÔNG lấy thêm

    def next(self):
        if not self.has_peeked:
            return self.iterator.next()                # không có bộ đệm: lấy thẳng
        result = self.peeked_element                   # có bộ đệm: trả bộ đệm và xóa nó
        self.has_peeked = False
        self.peeked_element = None
        return result

    def hasNext(self):
        return self.has_peeked or self.iterator.hasNext()


print(run_design(PeekingFlag, ex_ops, ex_args), "| kỳ vọng", ex_expected)

[None, 1, 2, 2, 3, False] | kỳ vọng [None, 1, 2, 2, 3, False]


In [11]:
# Code nộp LeetCode cho Cách 2 (Iterator do hệ thống định nghĩa sẵn)
class PeekingIterator:
    def __init__(self, iterator):
        """
        Initialize your data structure here.
        :type iterator: Iterator
        """
        self.iterator = iterator
        self.has_peeked = False
        self.peeked_element = None

    def peek(self):
        """
        Returns the next element in the iteration without advancing the iterator.
        :rtype: int
        """
        if not self.has_peeked:
            self.peeked_element = self.iterator.next()
            self.has_peeked = True
        return self.peeked_element

    def next(self):
        """
        :rtype: int
        """
        if not self.has_peeked:
            return self.iterator.next()
        result = self.peeked_element
        self.has_peeked = False
        self.peeked_element = None
        return result

    def hasNext(self):
        """
        :rtype: bool
        """
        return self.has_peeked or self.iterator.hasNext()

**Độ phức tạp Cách 2:** khởi tạo `O(1)` (**không** đọc iterator gốc), mỗi thao tác **`O(1)`**, **Space `O(1)`**. Hoạt động với iterator vô hạn.

### Cách 3 — Bộ đệm với đối tượng "rỗng" (sentinel): đúng với mọi kiểu dữ liệu

**Giải thích.** Thay `has_peeked` + `peeked_element` bằng **một** biến `self.peeked` mang `_EMPTY` khi không có gì. Phân biệt được "bộ đệm rỗng" với "bộ đệm chứa `None` hoặc `0`/`False`". Dùng `is` để so sánh với `_EMPTY`.

In [12]:
_EMPTY = object()


class PeekingSentinel:
    def __init__(self, iterator):
        self.iterator = iterator
        self.peeked = _EMPTY                         # chưa có gì trong bộ đệm

    def peek(self):
        if self.peeked is _EMPTY:
            self.peeked = self.iterator.next()
        return self.peeked

    def next(self):
        if self.peeked is _EMPTY:
            return self.iterator.next()
        value, self.peeked = self.peeked, _EMPTY
        return value

    def hasNext(self):
        return self.peeked is not _EMPTY or self.iterator.hasNext()


print(run_design(PeekingSentinel, ex_ops, ex_args), "| kỳ vọng", ex_expected)
generic = PeekingSentinel(Iterator([None, 0, False, "x"]))
print("dữ liệu tổng quát (None, 0, False, 'x'):", [generic.next() for _ in range(4)])

[None, 1, 2, 2, 3, False] | kỳ vọng [None, 1, 2, 2, 3, False]
dữ liệu tổng quát (None, 0, False, 'x'): [None, 0, False, 'x']


**Độ phức tạp Cách 3:** như Cách 2: `O(1)` mỗi thao tác, `O(1)` bộ nhớ phụ. Trả lời follow up: đúng với mọi kiểu dữ liệu, kể cả `None`.

### Cách 4 — Luôn lấy trước một phần tử (prefetch)

**Giải thích.** Giữ bất biến "**bộ đệm luôn chứa phần tử kế tiếp nếu còn**": ngay trong `__init__` và sau mỗi `next()` ta lấy sẵn phần tử kế từ iterator gốc (`_advance()`). Khi đó `peek()` chỉ trả bộ đệm; `next()` trả bộ đệm rồi `_advance()`; `hasNext()` là `bộ đệm không rỗng`. Mã đơn giản nhưng **lấy trước 1 phần tử ngay khi khởi tạo** (đọc sớm hơn Cách 2).

In [13]:
class PeekingPrefetch:
    def __init__(self, iterator):
        self.iterator = iterator
        self.peeked = _EMPTY
        self._advance()                                 # lấy trước phần tử đầu tiên

    def _advance(self):
        self.peeked = self.iterator.next() if self.iterator.hasNext() else _EMPTY

    def peek(self):
        return self.peeked

    def next(self):
        value = self.peeked
        self._advance()                                  # lấy trước phần tử kế
        return value

    def hasNext(self):
        return self.peeked is not _EMPTY


print(run_design(PeekingPrefetch, ex_ops, ex_args), "| kỳ vọng", ex_expected)

[None, 1, 2, 2, 3, False] | kỳ vọng [None, 1, 2, 2, 3, False]


**Độ phức tạp Cách 4:** khởi tạo `O(1)` (lấy một phần tử), mỗi thao tác `O(1)`, `O(1)` bộ nhớ phụ. Hơi **ít lười hơn** Cách 2 (đọc trước một phần tử dù người dùng chưa hỏi).

### Cách 5 — `deque` hỗ trợ xem trước `k` phần tử (`peek(k)`)

**Giải thích.** Mở rộng bộ đệm thành `deque`. `peek(k = 0)`: lấy từ iterator gốc cho tới khi bộ đệm có ít nhất `k + 1` phần tử rồi trả `buffer[k]`; `next()`: nếu bộ đệm không rỗng thì `popleft`, ngược lại lấy thẳng; `hasNext()`: bộ đệm không rỗng hoặc iterator gốc còn. Với `k = 0` là bài gốc; cùng cách làm cho các biến thể "xem trước nhiều phần tử".

In [14]:
class PeekingBuffered:
    def __init__(self, iterator):
        self.iterator = iterator
        self.buffer = deque()

    def peek(self, k=0):
        while len(self.buffer) <= k and self.iterator.hasNext():
            self.buffer.append(self.iterator.next())      # lấy thêm cho đủ k + 1 phần tử
        return self.buffer[k]

    def next(self):
        if self.buffer:
            return self.buffer.popleft()
        return self.iterator.next()

    def hasNext(self):
        return bool(self.buffer) or self.iterator.hasNext()


print(run_design(PeekingBuffered, ex_ops, ex_args), "| kỳ vọng", ex_expected)
pb = PeekingBuffered(Iterator([1, 2, 3, 4, 5]))
print("peek(0), peek(2), peek(4):", pb.peek(0), pb.peek(2), pb.peek(4), "| next:", pb.next(), pb.next())

[None, 1, 2, 2, 3, False] | kỳ vọng [None, 1, 2, 2, 3, False]
peek(0), peek(2), peek(4): 1 3 5 | next: 1 2


**Độ phức tạp Cách 5:** `peek(k)` `O(k)` (lấy thêm tối đa `k + 1` phần tử), `next` và `hasNext` `O(1)`; **Space `O(k)`** cho bộ đệm.

> Cách 2 là cách nộp bài. Cách 3 là phiên bản tổng quát cho mọi kiểu dữ liệu. Cách 4 và 5 là hai biến thể thiết kế. Cách 1 chỉ để đối chiếu.

## 8. TRACE từng bước

Ta theo dõi Cách 2 trên chuỗi thao tác của đề (in số lần iterator gốc bị gọi `next`), rồi Cách 5 với `peek(k)`.

In [15]:
def trace_ops(cls, nums, ops):
    inner = Iterator(nums)
    obj = cls(inner)
    print(f"nums={nums}")
    for op in ops:
        value = getattr(obj, op)()
        state = ""
        if hasattr(obj, "has_peeked"):
            state = f"bộ đệm: {'có ' + str(obj.peeked_element) if obj.has_peeked else 'rỗng'}"
        print(f"  {op:8} -> {value!s:6} | iterator gốc đã lấy {inner.next_calls} phần tử; {state}")


trace_ops(PeekingFlag, [1, 2, 3], ["next", "peek", "peek", "next", "next", "hasNext"])

nums=[1, 2, 3]
  next     -> 1      | iterator gốc đã lấy 1 phần tử; bộ đệm: rỗng
  peek     -> 2      | iterator gốc đã lấy 2 phần tử; bộ đệm: có 2
  peek     -> 2      | iterator gốc đã lấy 2 phần tử; bộ đệm: có 2
  next     -> 2      | iterator gốc đã lấy 2 phần tử; bộ đệm: rỗng
  next     -> 3      | iterator gốc đã lấy 3 phần tử; bộ đệm: rỗng
  hasNext  -> False  | iterator gốc đã lấy 3 phần tử; bộ đệm: rỗng


In [16]:
def trace_buffered():
    inner = Iterator([10, 20, 30, 40])
    pb = PeekingBuffered(inner)
    steps = [("peek(1)", lambda: pb.peek(1)), ("peek(0)", lambda: pb.peek(0)), ("next()", pb.next),
             ("peek(2)", lambda: pb.peek(2)), ("next()", pb.next), ("next()", pb.next), ("next()", pb.next), ("hasNext()", pb.hasNext)]
    for name, fn in steps:
        value = fn()
        print(f"  {name:10} -> {value!s:6} | bộ đệm = {list(pb.buffer)}, iterator gốc đã lấy {inner.next_calls}")


trace_buffered()

  peek(1)    -> 20     | bộ đệm = [10, 20], iterator gốc đã lấy 2
  peek(0)    -> 10     | bộ đệm = [10, 20], iterator gốc đã lấy 2
  next()     -> 10     | bộ đệm = [20], iterator gốc đã lấy 2
  peek(2)    -> 40     | bộ đệm = [20, 30, 40], iterator gốc đã lấy 4
  next()     -> 20     | bộ đệm = [30, 40], iterator gốc đã lấy 4
  next()     -> 30     | bộ đệm = [40], iterator gốc đã lấy 4
  next()     -> 40     | bộ đệm = [], iterator gốc đã lấy 4
  hasNext()  -> False  | bộ đệm = [], iterator gốc đã lấy 4


Bảng tóm tắt (Cách 2, `[1, 2, 3]`):

| thao tác | iterator gốc đã lấy | bộ đệm | trả về |
|---|---|---|---|
| `next()` | 1 | rỗng | `1` |
| `peek()` | 2 | có `2` | `2` |
| `peek()` (lần hai) | 2 (không lấy thêm) | có `2` | `2` |
| `next()` | 2 | rỗng | `2` |
| `next()` | 3 | rỗng | `3` |
| `hasNext()` | 3 | rỗng | `False` |

Lần `peek()` thứ hai không gọi `next` của iterator gốc: nếu có thì `2` bị mất và `3` bị bỏ qua.

## 9. Code nộp LeetCode

Bản nộp dùng Cách 2 (bộ đệm một phần tử kèm cờ). Đúng tên lớp `PeekingIterator`, đúng các phương thức `__init__(self, iterator)`, `peek(self)`, `next(self)`, `hasNext(self)`.

In [17]:
class PeekingIterator:
    def __init__(self, iterator):
        """
        Initialize your data structure here.
        :type iterator: Iterator
        """
        self.iterator = iterator
        self.has_peeked = False            # bộ đệm có chứa phần tử đã lấy trước không?
        self.peeked_element = None

    def peek(self):
        """
        Returns the next element in the iteration without advancing the iterator.
        :rtype: int
        """
        if not self.has_peeked:            # chưa lấy trước: lấy một lần từ iterator gốc
            self.peeked_element = self.iterator.next()
            self.has_peeked = True
        return self.peeked_element         # các lần peek sau trả lại bộ đệm

    def next(self):
        """
        :rtype: int
        """
        if not self.has_peeked:
            return self.iterator.next()    # không có bộ đệm: lấy thẳng
        result = self.peeked_element       # có bộ đệm: trả bộ đệm rồi xóa
        self.has_peeked = False
        self.peeked_element = None
        return result

    def hasNext(self):
        """
        :rtype: bool
        """
        return self.has_peeked or self.iterator.hasNext()


assert run_design(PeekingIterator, ex_ops, ex_args) == ex_expected
print("PASS ví dụ đề cho class PeekingIterator")

PASS ví dụ đề cho class PeekingIterator


## 10. Vì sao thuật toán đúng

- **Bất biến:** nếu `has_peeked` là `True` thì `peeked_element` là phần tử **kế tiếp** của dãy đối với người dùng, và phần tử đó đã bị lấy khỏi iterator gốc; các phần tử còn lại trong iterator gốc đứng **sau** nó. Nếu `has_peeked` là `False` thì phần tử kế tiếp (nếu có) là cái iterator gốc sẽ trả.
- **`peek()`:** nếu `has_peeked` đã đúng thì trả bộ đệm (không lấy thêm); nếu chưa, lấy một phần tử từ iterator gốc và đặt bộ đệm: bất biến thỏa. `peek()` không làm "người dùng" tiến lên vì không xóa bộ đệm.
- **`next()`:** nếu có bộ đệm, phần tử kế tiếp chính là bộ đệm: trả và xóa (con trỏ tiến lên một phần tử); nếu không, phần tử kế tiếp là cái iterator gốc trả: trả thẳng. Bất biến giữ.
- **`hasNext()`:** còn phần tử khi và chỉ khi bộ đệm có phần tử hoặc iterator gốc còn phần tử.
- **Không mất, không trùng phần tử:** mỗi phần tử bị lấy khỏi iterator gốc **đúng một lần** và được trả `next()` **đúng một lần** (hoặc nằm trong bộ đệm chờ).
- **Mọi kiểu dữ liệu:** Cách 3 dùng `_EMPTY` (đối tượng riêng) nên không nhầm "chưa peek" với dữ liệu `None`.
- **Cách 4:** bất biến "bộ đệm luôn là phần tử kế (hoặc `_EMPTY` khi hết)"; `_advance` sau mỗi `next` giữ đúng. **Cách 5:** bộ đệm là tiền tố của phần còn lại của dãy.

## 11. Phân tích Time / Space Complexity

Gọi `n` là số phần tử của iterator gốc.

- **Cách 1 (lấy hết vào list):** khởi tạo `O(n)`, mỗi thao tác `O(1)`, Space `O(n)`; không dùng được cho dãy vô hạn.
- **Cách 2 (bộ đệm một phần tử + cờ):** mỗi thao tác `O(1)`, Space `O(1)`.
- **Cách 3 (sentinel):** mỗi thao tác `O(1)`, Space `O(1)`; đúng cho mọi kiểu dữ liệu.
- **Cách 4 (prefetch):** khởi tạo `O(1)`, mỗi thao tác `O(1)`, Space `O(1)`.
- **Cách 5 (`deque`, `peek(k)`):** `peek(k)` `O(k)`, `next` và `hasNext` `O(1)`, Space `O(k)`.
- Cách 2–4 chỉ lấy trước **tối đa một** phần tử, nên số lần gọi `Iterator.next()` luôn `≤` số phần tử đã được tiêu thụ + 1.

## 12. Bộ test

Ta kiểm tra theo ba cách: (1) ví dụ của đề; (2) **mô hình độc lập** (`ModelIterator`: list + con trỏ, không dùng iterator gốc) chạy chuỗi thao tác ngẫu nhiên hợp lệ có seed; (3) **tính lười**: với các cách 2–4, số lần gọi `Iterator.next()` không vượt số phần tử đã tiêu thụ cộng một.

In [18]:
impls = {
    "eager": PeekingEager,
    "flag": PeekingFlag,
    "sentinel": PeekingSentinel,
    "prefetch": PeekingPrefetch,
    "buffered": PeekingBuffered,
    "submit": PeekingIterator,
}
LAZY = ("flag", "sentinel", "buffered", "submit")           # prefetch đọc trước 1 phần tử ngay khi khởi tạo


class ModelIterator:
    """Mô hình độc lập: list + con trỏ."""

    def __init__(self, nums):
        self.nums, self.pos = list(nums), 0

    def peek(self):
        return self.nums[self.pos]

    def next(self):
        v = self.nums[self.pos]
        self.pos += 1
        return v

    def hasNext(self):
        return self.pos < len(self.nums)


def random_ops(model_nums, n_ops, allow_peek=True):
    """Chuỗi thao tác hợp lệ: next/peek chỉ khi còn phần tử; hasNext bất kỳ lúc nào."""
    model = ModelIterator(model_nums)
    ops = []
    for _ in range(n_ops):
        choices = ["hasNext"]
        if model.hasNext():
            choices.append("next")
            if allow_peek:
                choices += ["peek", "peek"]
        op = random.choice(choices)
        ops.append(op)
        if op == "next":
            model.next()
    return ops


def replay(cls, nums, ops):
    inner = Iterator(nums)
    obj = cls(inner)
    return [getattr(obj, op)() for op in ops], inner.next_calls


def replay_model(nums, ops):
    model = ModelIterator(nums)
    return [getattr(model, op)() for op in ops]


for name, cls in impls.items():
    assert run_design(cls, ex_ops, ex_args) == ex_expected, name
print("PASS ví dụ của đề (mọi lớp)")

PASS ví dụ của đề (mọi lớp)


### 12.1. Các trường hợp biên

In [19]:
def check_ops(nums, ops, label, names=None):
    expected = replay_model(nums, ops)
    for name, cls in impls.items():
        if names is not None and name not in names:
            continue
        got, _ = replay(cls, nums, ops)
        assert got == expected, (label, name, got, expected)
    print("PASS", label, "->", expected)


check_ops([5], ["hasNext", "peek", "peek", "next", "hasNext"], "n = 1")
check_ops([1, 2, 3], ["peek", "peek", "peek", "next", "next", "next", "hasNext"], "peek lặp lại không làm tiến lên")
check_ops([1, 2, 3], ["next", "peek", "next", "next", "hasNext"], "ví dụ 1 của đề")
check_ops([1, 2], ["hasNext", "hasNext", "next", "hasNext", "next", "hasNext"], "hasNext lặp lại")
check_ops([7, 8, 9], ["peek", "next", "peek", "next", "peek", "next", "hasNext"], "peek rồi next xen kẽ")
check_ops([1, 2, 3, 4], ["next", "next", "next", "peek", "hasNext", "next", "hasNext"], "peek phần tử cuối")
check_ops(list(range(1, 1001)), ["peek", "next"] * 500, "n = 1000, peek/next xen kẽ")
check_ops(list(range(1, 1001)), ["next"] * 1000 + ["hasNext"], "chỉ next đến hết, rồi hasNext")

PASS n = 1 -> [True, 5, 5, 5, False]
PASS peek lặp lại không làm tiến lên -> [1, 1, 1, 1, 2, 3, False]
PASS ví dụ 1 của đề -> [1, 2, 2, 3, False]
PASS hasNext lặp lại -> [True, True, 1, True, 2, False]
PASS peek rồi next xen kẽ -> [7, 7, 8, 8, 9, 9, False]
PASS peek phần tử cuối -> [1, 2, 3, 4, True, 4, False]
PASS n = 1000, peek/next xen kẽ -> [1, 1, 2, 2, 3, 3, 4, 4, 5, 5, 6, 6, 7, 7, 8, 8, 9, 9, 10, 10, 11, 11, 12, 12, 13, 13, 14, 14, 15, 15, 16, 16, 17, 17, 18, 18, 19, 19, 20, 20, 21, 21, 22, 22, 23, 23, 24, 24, 25, 25, 26, 26, 27, 27, 28, 28, 29, 29, 30, 30, 31, 31, 32, 32, 33, 33, 34, 34, 35, 35, 36, 36, 37, 37, 38, 38, 39, 39, 40, 40, 41, 41, 42, 42, 43, 43, 44, 44, 45, 45, 46, 46, 47, 47, 48, 48, 49, 49, 50, 50, 51, 51, 52, 52, 53, 53, 54, 54, 55, 55, 56, 56, 57, 57, 58, 58, 59, 59, 60, 60, 61, 61, 62, 62, 63, 63, 64, 64, 65, 65, 66, 66, 67, 67, 68, 68, 69, 69, 70, 70, 71, 71, 72, 72, 73, 73, 74, 74, 75, 75, 76, 76, 77, 77, 78, 78, 79, 79, 80, 80, 81, 81, 82, 82, 83, 83, 84, 84

### 12.2. Tính chất và phản ví dụ

In [20]:
random.seed(284)

# Tính chất 1: mọi chuỗi thao tác ngẫu nhiên hợp lệ khớp mô hình độc lập
for _ in range(500):
    nums = [random.randint(1, 1000) for _ in range(random.randint(1, 20))]
    ops = random_ops(nums, random.randint(1, 60))
    expected = replay_model(nums, ops)
    for name, cls in impls.items():
        got, _ = replay(cls, nums, ops)
        assert got == expected, (name, nums, ops)
print("PASS tính chất: 500 chuỗi thao tác ngẫu nhiên khớp mô hình độc lập")

# Tính chất 2 (tính lười): với các cách 2, 3, 5 số lần iterator gốc bị lấy <= số phần tử đã tiêu thụ + 1
for _ in range(300):
    nums = [random.randint(1, 1000) for _ in range(random.randint(1, 20))]
    ops = random_ops(nums, random.randint(1, 40))
    consumed = sum(op == "next" for op in ops)
    for name in LAZY:
        _, calls = replay(impls[name], nums, ops)
        assert calls <= consumed + 1, (name, calls, consumed)
print("PASS tính chất: các cách lười lấy tối đa 'đã tiêu thụ + 1' phần tử từ iterator gốc")

# Tính chất 3: lớp bọc hoạt động với iterator VÔ HẠN (Cách 1 không dùng được)
for name in ("flag", "sentinel", "prefetch", "buffered"):
    inf = Naturals()
    p = impls[name](inf)
    assert p.hasNext() and p.peek() == 0 and p.next() == 0 and p.peek() == 1 and p.next() == 1
    assert inf.next_calls <= 3
print("PASS tính chất: các cách bộ đệm chạy được trên iterator vô hạn (số lần lấy <= 3)")

# Tính chất 4: tổng quát mọi kiểu dữ liệu (kể cả None, 0, False, chuỗi rỗng)
data = [None, 0, False, "", "x", None]
for name in ("sentinel", "prefetch", "buffered", "eager"):
    p = impls[name](Iterator(data))
    out = []
    while p.hasNext():
        assert p.peek() is p.peek()
        out.append(p.next())
    assert out == data, (name, out)
print("PASS tính chất: dữ liệu tổng quát (None, 0, False, '') đúng ở cách dùng sentinel/prefetch/buffered/eager")

PASS tính chất: 500 chuỗi thao tác ngẫu nhiên khớp mô hình độc lập
PASS tính chất: các cách lười lấy tối đa 'đã tiêu thụ + 1' phần tử từ iterator gốc
PASS tính chất: các cách bộ đệm chạy được trên iterator vô hạn (số lần lấy <= 3)
PASS tính chất: dữ liệu tổng quát (None, 0, False, '') đúng ở cách dùng sentinel/prefetch/buffered/eager


### 12.3. Đối chiếu ngẫu nhiên (có seed) với mô hình độc lập

In [21]:
random.seed(2026)
for trial in range(3000):
    nums = [random.randint(1, 1000) for _ in range(random.randint(1, 30))]
    ops = random_ops(nums, random.randint(1, 80))
    expected = replay_model(nums, ops)
    for name, cls in impls.items():
        got, _ = replay(cls, nums, ops)
        assert got == expected, (name, nums, ops, got, expected)

# chuỗi thao tác dài theo đề: n = 1000 phần tử, 1000 lời gọi
for trial in range(20):
    nums = [random.randint(1, 1000) for _ in range(1000)]
    ops = random_ops(nums, 1000)
    expected = replay_model(nums, ops)
    for name, cls in impls.items():
        assert replay(cls, nums, ops)[0] == expected, name
print("PASS 3000 chuỗi thao tác ngẫu nhiên + 20 chuỗi dài 1000 lời gọi: mọi lớp khớp mô hình độc lập")

PASS 3000 chuỗi thao tác ngẫu nhiên + 20 chuỗi dài 1000 lời gọi: mọi lớp khớp mô hình độc lập


## 13. Các lỗi người mới hay gặp

### Lỗi 1: `peek()` gọi `next()` của iterator gốc mỗi lần

In [22]:
class SaiPeekMoiLan:
    def __init__(self, iterator):
        self.iterator = iterator

    def peek(self):
        return self.iterator.next()                 # sai: mỗi lần peek làm iterator gốc tiến lên

    def next(self):
        return self.iterator.next()

    def hasNext(self):
        return self.iterator.hasNext()


p = SaiPeekMoiLan(Iterator([1, 2, 3]))
out = [p.next(), p.peek(), p.next()]
print("next, peek, next ->", out, "| đúng: [1, 2, 2]")

next, peek, next -> [1, 2, 3] | đúng: [1, 2, 2]


**Vì sao sai:** `peek` chỉ được **xem**; gọi `iterator.next()` làm phần tử tiến lên và bị mất. Với `[1,2,3]`: `next → 1`, `peek` lấy `2` (và `2` bị mất), rồi `next` trả `3` thay vì `2`. Phải lấy một lần rồi **cất vào bộ đệm**.

### Lỗi 2: `next()` bỏ qua bộ đệm

In [23]:
class SaiNextBoQuaDem:
    def __init__(self, iterator):
        self.iterator = iterator
        self.has_peeked = False
        self.peeked_element = None

    def peek(self):
        if not self.has_peeked:
            self.peeked_element = self.iterator.next()
            self.has_peeked = True
        return self.peeked_element

    def next(self):
        return self.iterator.next()                  # sai: không xét bộ đệm

    def hasNext(self):
        return self.has_peeked or self.iterator.hasNext()


p = SaiNextBoQuaDem(Iterator([1, 2, 3]))
out = [p.next(), p.peek(), p.next()]
print("sai:", out, "| đúng: [1, 2, 2]")

sai: [1, 2, 3] | đúng: [1, 2, 2]


**Vì sao sai:** sau `peek()` phần tử `2` nằm trong bộ đệm (đã bị lấy khỏi iterator gốc). `next()` phải trả `2` từ bộ đệm; nếu hỏi iterator gốc thì được `3` và `2` bị mất.

### Lỗi 3: `hasNext()` bỏ qua bộ đệm

In [24]:
class SaiHasNextBoQuaDem:
    def __init__(self, iterator):
        self.iterator = iterator
        self.has_peeked = False
        self.peeked_element = None

    def peek(self):
        if not self.has_peeked:
            self.peeked_element = self.iterator.next()
            self.has_peeked = True
        return self.peeked_element

    def next(self):
        if self.has_peeked:
            self.has_peeked = False
            return self.peeked_element
        return self.iterator.next()

    def hasNext(self):
        return self.iterator.hasNext()               # sai: bỏ qua phần tử đang nằm trong bộ đệm


p = SaiHasNextBoQuaDem(Iterator([1, 2]))
p.next()
p.peek()                                              # phần tử 2 đã bị lấy khỏi iterator gốc, đang ở bộ đệm
print("hasNext() sai:", p.hasNext(), "| đúng: True")

hasNext() sai: False | đúng: True


**Vì sao sai:** sau khi `peek` lấy hết phần tử cuối vào bộ đệm, iterator gốc trả `hasNext() = False` dù người dùng vẫn chưa nhận phần tử đó. Phải trả `has_peeked or iterator.hasNext()`.

### Lỗi 4: Dùng `None` làm dấu "chưa peek"

In [25]:
class SaiDungNone:
    def __init__(self, iterator):
        self.iterator = iterator
        self.peeked = None                            # sai: dữ liệu thật cũng có thể là None

    def peek(self):
        if self.peeked is None:
            self.peeked = self.iterator.next()
        return self.peeked

    def next(self):
        if self.peeked is None:
            return self.iterator.next()
        value, self.peeked = self.peeked, None
        return value

    def hasNext(self):
        return self.peeked is not None or self.iterator.hasNext()


p = SaiDungNone(Iterator([None, 5]))
print("peek, peek ->", [p.peek(), p.peek()], "| đúng: [None, None]")

peek, peek -> [None, 5] | đúng: [None, None]


**Vì sao sai:** phần tử `None` được peek vào bộ đệm nhưng code coi `peeked is None` là "chưa có": lần `peek` thứ hai lại lấy thêm từ iterator gốc (được `5`), làm mất phần tử `None` và `peek` không còn nhất quán. Dùng cờ boolean hoặc đối tượng `_EMPTY` riêng (Cách 3) để hỗ trợ **mọi kiểu dữ liệu**.

## 14. Mẹo làm nhanh & nhận dạng pattern

- Muốn "xem trước mà không lấy" trên một nguồn chỉ cho "lấy": **lấy một phần tử ra và cất vào bộ đệm**; mọi thao tác phải **ưu tiên bộ đệm**.
- Ba thao tác, ba câu hỏi: `peek` (bộ đệm trống thì lấy thêm), `next` (có bộ đệm thì dùng), `hasNext` (bộ đệm hoặc nguồn còn).
- Dùng `_EMPTY = object()` (hoặc cờ) làm dấu "rỗng" để đúng với mọi kiểu dữ liệu.
- Giữ tính **lười** (không đọc trước nhiều hơn cần) để chạy được trên nguồn vô hạn hoặc đắt.
- Cần xem trước `k` phần tử: bộ đệm `deque`.
- Test bài thiết kế: **mô hình độc lập** (list + con trỏ) và chuỗi thao tác ngẫu nhiên có seed; kiểm tra thêm số lần gọi nguồn.

## 15. Biến thể của bài

1. **`peek(k)`** (xem trước `k` phần tử): Cách 5.
2. **`unread()`**: trả lại phần tử vừa `next()` (hoàn tác một bước).
3. **Iterator hỗ trợ `skip(n)`** (bỏ `n` phần tử).
4. **Peeking cho mọi kiểu dữ liệu** (đã có, dùng sentinel).
5. **Iterator lọc** (chỉ trả phần tử thỏa điều kiện) có `peek`.

In [26]:
class PeekUnread:
    """Biến thể 2: next() có thể hoàn tác một bước bằng unread()."""

    def __init__(self, iterator):
        self.iterator = iterator
        self.buffer = deque()
        self.last = _EMPTY

    def peek(self):
        if not self.buffer:
            self.buffer.append(self.iterator.next())
        return self.buffer[0]

    def next(self):
        value = self.buffer.popleft() if self.buffer else self.iterator.next()
        self.last = value
        return value

    def hasNext(self):
        return bool(self.buffer) or self.iterator.hasNext()

    def unread(self):
        assert self.last is not _EMPTY
        self.buffer.appendleft(self.last)
        self.last = _EMPTY


class PeekSkip(PeekingBuffered):
    """Biến thể 3."""

    def skip(self, n):
        for _ in range(n):
            if self.hasNext():
                self.next()


class FilteredPeeking:
    """Biến thể 5: chỉ trả phần tử thỏa pred, có peek."""

    def __init__(self, iterator, pred):
        self.inner = PeekingSentinel(iterator)
        self.pred = pred

    def _skip(self):
        while self.inner.hasNext() and not self.pred(self.inner.peek()):
            self.inner.next()

    def peek(self):
        self._skip()
        return self.inner.peek()

    def next(self):
        self._skip()
        return self.inner.next()

    def hasNext(self):
        self._skip()
        return self.inner.hasNext()


pu = PeekUnread(Iterator([1, 2, 3]))
assert [pu.next(), pu.next()] == [1, 2]
pu.unread()
assert [pu.peek(), pu.next(), pu.next(), pu.hasNext()] == [2, 2, 3, False]
ps = PeekSkip(Iterator(range(10)))
ps.skip(4)
assert ps.peek() == 4 and ps.next() == 4
fp = FilteredPeeking(Iterator([1, 2, 3, 4, 5, 6]), lambda x: x % 2 == 0)
assert [fp.peek(), fp.next(), fp.peek(), fp.next(), fp.next(), fp.hasNext()] == [2, 2, 4, 4, 6, False]
print("PASS biến thể: unread, skip, iterator lọc có peek")

PASS biến thể: unread, skip, iterator lọc có peek


## 16. So sánh các cách làm

| Cách | Ý tưởng | Khởi tạo | Mỗi thao tác | Space | Khi nào dùng |
|---|---|---|---|---|---|
| 1. Lấy hết vào list | `items`, con trỏ | `O(n)` | `O(1)` | `O(n)` | đơn giản; **không** dùng cho dãy vô hạn |
| 2. Bộ đệm 1 phần tử + cờ | `has_peeked`, `peeked_element` | `O(1)` | `O(1)` | `O(1)` | **cách nộp bài** |
| 3. Bộ đệm + sentinel | `_EMPTY` | `O(1)` | `O(1)` | `O(1)` | **mọi kiểu dữ liệu** (follow up) |
| 4. Prefetch | luôn giữ phần tử kế trong bộ đệm | `O(1)` (lấy 1 phần tử) | `O(1)` | `O(1)` | mã gọn, nhưng đọc sớm 1 phần tử |
| 5. `deque` | `peek(k)` | `O(1)` | `peek(k)`: `O(k)` | `O(k)` | xem trước nhiều phần tử |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Với `p = PeekingIterator(Iterator([4, 5]))`, đoán kết quả của `p.peek(), p.peek(), p.next(), p.hasNext(), p.next(), p.hasNext()`.

**Bài 2 (tính tay).** Với `Iterator([7, 8, 9])` và chuỗi `peek, next, peek, hasNext, next, next, hasNext`: lập bảng (thao tác, bộ đệm, số phần tử iterator gốc đã bị lấy, trả về).

**Bài 3 (điền TODO).** Hoàn thành `class PeekingTodo` theo Cách 2.

In [27]:
class PeekingTodo:
    def __init__(self, iterator):
        # TODO 1: lưu iterator, has_peeked = False, peeked_element = None
        pass

    def peek(self):
        # TODO 2: nếu chưa peek thì lấy iterator.next() cất vào peeked_element và has_peeked = True; trả peeked_element
        pass

    def next(self):
        # TODO 3: nếu has_peeked thì trả peeked_element và xóa bộ đệm; ngược lại trả iterator.next()
        pass

    def hasNext(self):
        # TODO 4: trả has_peeked or iterator.hasNext()
        pass

# KHÔNG tạo đối tượng / gọi hàm TODO ở đây để khi Run All không bị lỗi (các TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `class PeekingBuffered2` theo Cách 5 (hỗ trợ `peek(k=0)`) không nhìn code mẫu. Kiểm tra bằng ví dụ của đề và `peek(3)` trên `Iterator([1,2,3,4,5])`.

**Bài 5 (biến thể / mini challenge).** (a) Viết `class PeekingLast` thêm phương thức `last()` trả về phần tử **vừa được `next()` trả** gần nhất (hoặc `None` nếu chưa có). (b) Viết `class PeekingTake` nhận thêm giới hạn `limit` và chỉ cho `next()` tối đa `limit` phần tử (sau đó `hasNext()` là `False`), vẫn hỗ trợ `peek`.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `peek` không làm tiến lên; `peek` lặp lại trả cùng giá trị. Bài 2: ghi lại iterator gốc đã bị lấy bao nhiêu phần tử sau mỗi bước.
- **Mức 2 (vừa):** Bài 1: `4, 4, 4, True, 5, False`. Bài 2: `peek` → `7` (gốc đã lấy 1); `next` → `7` (xóa bộ đệm); `peek` → `8` (gốc đã lấy 2); `hasNext` → `True`; `next` → `8`; `next` → `9` (gốc đã lấy 3); `hasNext` → `False`. Bài 3: bốn TODO trong comment.
- **Mức 3 (sâu):** Bài 4: `while len(buf) <= k and iterator.hasNext(): buf.append(iterator.next())`; trả `buf[k]`. Bài 5(a): lưu `self._last` mỗi khi `next()` trả phần tử. Bài 5(b): đếm `self.taken`; `hasNext` là `taken < limit and (bộ đệm hoặc iterator còn)`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [28]:
# Bài 1
p = PeekingIterator(Iterator([4, 5]))
assert [p.peek(), p.peek(), p.next(), p.hasNext(), p.next(), p.hasNext()] == [4, 4, 4, True, 5, False]

# Bài 2
assert replay(PeekingIterator, [7, 8, 9], ["peek", "next", "peek", "hasNext", "next", "next", "hasNext"]) == ([7, 7, 8, True, 8, 9, False], 3)

# Bài 3
class PeekingTodoAns:
    def __init__(self, iterator):
        self.iterator = iterator
        self.has_peeked = False
        self.peeked_element = None

    def peek(self):
        if not self.has_peeked:
            self.peeked_element = self.iterator.next()
            self.has_peeked = True
        return self.peeked_element

    def next(self):
        if self.has_peeked:
            self.has_peeked = False
            value, self.peeked_element = self.peeked_element, None
            return value
        return self.iterator.next()

    def hasNext(self):
        return self.has_peeked or self.iterator.hasNext()

assert run_design(PeekingTodoAns, ex_ops, ex_args) == ex_expected

# Bài 4
class PeekingBuffered2:
    def __init__(self, iterator):
        self.iterator = iterator
        self.buf = deque()

    def peek(self, k=0):
        while len(self.buf) <= k and self.iterator.hasNext():
            self.buf.append(self.iterator.next())
        return self.buf[k]

    def next(self):
        return self.buf.popleft() if self.buf else self.iterator.next()

    def hasNext(self):
        return bool(self.buf) or self.iterator.hasNext()

assert run_design(PeekingBuffered2, ex_ops, ex_args) == ex_expected
pb2 = PeekingBuffered2(Iterator([1, 2, 3, 4, 5]))
assert pb2.peek(3) == 4 and pb2.next() == 1 and pb2.peek(3) == 5

# Bài 5(a), 5(b)
class PeekingLast(PeekingSentinel):
    _last = None

    def next(self):
        self._last = super().next()
        return self._last

    def last(self):
        return self._last

class PeekingTake(PeekingSentinel):
    def __init__(self, iterator, limit):
        super().__init__(iterator)
        self.limit, self.taken = limit, 0

    def next(self):
        self.taken += 1
        return super().next()

    def hasNext(self):
        return self.taken < self.limit and super().hasNext()

pl = PeekingLast(Iterator([1, 2, 3]))
assert pl.last() is None and pl.next() == 1 and pl.last() == 1 and pl.peek() == 2 and pl.last() == 1
pt = PeekingTake(Iterator([1, 2, 3, 4]), 2)
out = []
while pt.hasNext():
    v = pt.peek()
    assert v == pt.next()
    out.append(v)
assert out == [1, 2] and pt.taken == 2
print("PASS đáp án phần 19")

PASS đáp án phần 19


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (bộ đệm một phần tử + cờ)

```python
class PeekingIterator:
    def __init__(self, iterator):
        self.iterator = iterator
        self.has_peeked = False
        self.peeked_element = None

    def peek(self):
        if not self.has_peeked:
            self.peeked_element = self.iterator.next()
            self.has_peeked = True
        return self.peeked_element

    def next(self):
        if not self.has_peeked:
            return self.iterator.next()
        result = self.peeked_element
        self.has_peeked = False
        self.peeked_element = None
        return result

    def hasNext(self):
        return self.has_peeked or self.iterator.hasNext()
```

**Checklist**

- [ ] `peek()` lấy từ iterator gốc **một lần** rồi cất vào bộ đệm; lần sau trả bộ đệm (không lấy thêm)?
- [ ] `next()` **ưu tiên bộ đệm** (trả và xóa) trước khi hỏi iterator gốc?
- [ ] `hasNext()` = `has_peeked or iterator.hasNext()` (không bỏ qua bộ đệm)?
- [ ] Không đọc iterator gốc trong `__init__` (giữ tính lười, chạy được cho dãy vô hạn)?
- [ ] Không dùng `None` làm dấu "chưa peek" nếu dữ liệu có thể là `None` (dùng cờ hoặc `_EMPTY`)?
- [ ] Đã thử: `peek` lặp lại, `peek` rồi `hasNext` ở phần tử cuối, `hasNext` lặp lại, chuỗi 1000 lời gọi?
- [ ] Đã đối chiếu với mô hình độc lập (list + con trỏ) trên chuỗi thao tác ngẫu nhiên có seed?
- [ ] Skeleton đúng `class PeekingIterator` / `__init__(self, iterator)` / `peek` / `next` / `hasNext`?